# SI26-Week2-Humna — Urdu OCR Project

**What this notebook does:** This notebook preprocesses all 153 raw Urdu images collected in Week 1
(books, newspaper clippings, synthetic text, and UTRSet samples), standardising them into
`data/processed/`. It then runs baseline Tesseract OCR (`lang='urd'`) on five representative images
— one from each source type — and compares Tesseract's output against the real ground-truth text
from `data/labels.csv` to see exactly how and why it fails on Urdu.

**Note:** This notebook lives in `SI26-Week2/` while the raw data lives in `SI26-Week1/data/`.
The setup cell below locates that data automatically, so this works whether you run it from
the repo root or from inside `SI26-Week2/` (e.g. in a GitHub Codespace).

### Setup — locate the Week 1 data

In [1]:
import os

# Works whether this notebook is run from the repo root or from inside SI26-Week2/
if os.path.exists('SI26-Week1/data/raw'):
    BASE = 'SI26-Week1'
elif os.path.exists('../SI26-Week1/data/raw'):
    BASE = '../SI26-Week1'
else:
    raise FileNotFoundError(
        'Could not locate SI26-Week1/data/raw. Check that this notebook sits either at the '
        'repo root or inside SI26-Week2/, next to the SI26-Week1 folder.'
    )

RAW_DIR = f'{BASE}/data/raw'
LABELS_CSV = f'{BASE}/data/labels.csv'
print(f'Using data from: {RAW_DIR}')

Using data from: ../SI26-Week1/data/raw


## Part A — Preprocess Your Images
### Step 2 — Install Libraries

In [2]:
!pip install -q opencv-python-headless pillow matplotlib

import cv2
import numpy as np
from PIL import Image
import glob
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Libraries loaded successfully!


### Step 3 — Write Your Preprocessing Function

In [3]:
def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return

    # Step 1: Convert to grayscale (removes colour noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to standard size (keeps all images same dimensions)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise (makes text cleaner)
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Binarise (make pixels either pure black or pure white)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary

# Create output folder (relative to wherever this notebook is running from)
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [4]:
# Find all images under the Week 1 raw data folder
all_images = glob.glob(f'{RAW_DIR}/**/*.jpg', recursive=True)
all_images += glob.glob(f'{RAW_DIR}/**/*.png', recursive=True)
print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in sorted(all_images):
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')

Found 153 images to process
Done! Processed 153 images
Check data/processed/ folder


## Part B — Test Tesseract OCR on Your Urdu Images

Five images were selected — one from each source type collected in Week 1
(**books**, **newspaper**, **synthetic**, and two from **UTRSet / other**, since it's the largest
category) — to see how Tesseract's Urdu model handles real, varied input.

**Codespaces note:** installing system packages (`tesseract-ocr`) needs `sudo` in a Codespace —
plain `apt-get install` fails with a `dpkg lock ... Permission denied` error, which is what the
`sudo` below avoids. The `which tesseract` check skips the (slow) install step entirely if it's
already installed.

In [5]:
# Install Tesseract + Urdu language pack (skips cleanly if already installed)
!which tesseract > /dev/null || (sudo apt-get update -qq && sudo apt-get install -y tesseract-ocr tesseract-ocr-urd)
!pip install -q pytesseract

import pytesseract
import csv

# Load ground-truth labels collected in Week 1
labels = {}
with open(LABELS_CSV, encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            labels[row[0]] = row[1]

# One representative image per source type (category, filename)
test_set = [
    ('books', 'book_001.png'),
    ('newspaper', 'newspaper_001.png'),
    ('synthetic', 'urdu_1.png'),
    ('other', 'utrset_000.png'),
    ('other', 'utrset_005.png'),
]

print('=== Tesseract Results on Urdu Images ===')
print()
for category, filename in test_set:
    label_key = f'data/raw/{category}/{filename}'   # matches labels.csv path format
    processed_path = f'data/processed/{filename}'
    img = Image.open(processed_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {label_key}')
    print(f'Ground truth   : {labels.get(label_key)}')
    print(f'Tesseract output: {result!r}')
    print('---')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  liblept5 libopenjp2-7 libtesseract5 libwebpmux3 tesseract-ocr-eng
  tesseract-ocr-osd
The following NEW packages will be installed:
  liblept5 libopenjp2-7 libtesseract5 libwebpmux3 tesseract-ocr
  tesseract-ocr-eng tesseract-ocr-osd tesseract-ocr-urd
0 upgraded, 8 newly installed, 0 to remove and 127 not upgraded.
Need to get 9605 kB of archives.
After this operation, 23.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libopenjp2-7 amd64 2.5.0-2ubuntu0.5 [174 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libwebpmux3 amd64 1.3.2-0.4build3 [25.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 liblept5 amd64 1.82.0-3build4 [1099 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/universe amd64 libtesseract5 amd64 5.3.4-1build5 [1291 kB]
Get:5 http://arc

## Step 4 — Gap Analysis

### Image 1: `data/raw/books/book_001.png`
- **Actual Urdu text:** دیباچہ ("Preface")
- **Tesseract output:** *(empty — nothing detected)*
- **What went wrong:** Total failure, not just wrong characters. The original scan is a tall
  portrait page (510×706 px). Our preprocessing step force-resizes every image to a fixed
  512×128 canvas, which squashes a portrait page to about a fifth of its natural height. The
  text is crushed into an unreadable smear before Tesseract ever sees it, so it returns nothing.

### Image 2: `data/raw/newspaper/newspaper_001.png`
- **Actual Urdu text:** پہلی بات ("The First Word" / foreword)
- **Tesseract output:** *(empty — nothing detected)*
- **What went wrong:** Same root cause as Image 1 — a 466×705 portrait newspaper clipping
  squashed into 512×128. The vertical compression destroys the letterforms before OCR can even
  attempt segmentation.

### Image 3: `data/raw/synthetic/urdu_1.png`
- **Actual Urdu text:** پاکستان زندہ باد ("Long live Pakistan")
- **Tesseract output:** پالسحا نے 2عدہ یاھ
- **What went wrong:** This image's original aspect ratio (229×119) was already close to the
  512×128 target, so the resize wasn't as destructive — Tesseract did produce output this time.
  But it's still gibberish: no real word is recognised correctly, letters are swapped or merged,
  and a phantom digit "2" appears out of nowhere.

### Image 4: `data/raw/other/utrset_000.png`
- **Actual Urdu text:** اندراج و تحریر شرعاً صرف مستحب اور پسندیدہ ہے وہ واجب نہیں کہ کسی شرعی (13 words)
- **Tesseract output:** ا ام سپ لا (4 disconnected fragments)
- **What went wrong:** Almost the entire sentence is missing. Of ~13 real words, Tesseract
  returned 4 short fragments that don't correspond to any actual Urdu word — a sign it couldn't
  segment the connected script into meaningful characters at all.

### Image 5: `data/raw/other/utrset_005.png`
- **Actual Urdu text:** مقدمات سے نجات مل سکتی ہے، نکاح کے ثبوت اورک دین مہر کے تعین میں سہولت ہوتی (14 words)
- **Tesseract output:** ااع للا 2 کے
- **What went wrong:** Nearly total word loss again, meaningless fragments, and — like Image 3 —
  a hallucinated "2" that doesn't exist anywhere in the source text.

### Summary

**Tesseract fails on Urdu because** the script it's actually tuned for is Naskh-style printed
Arabic, not the Nastaliq calligraphic style that almost all real Urdu text — books, newspapers,
and the UTRSet samples alike — is set in. Nastaliq is diagonal and cursive: letters slope
downward across the line instead of sitting on a flat baseline, each letter changes shape
depending on whether it's isolated or in the initial, medial, or final position within a word,
and neighbouring letters overlap and stack vertically rather than staying cleanly separated.
Tesseract's segmentation logic assumes clean, mostly-horizontal word boundaries, so on dense
Nastaliq lines it either merges separate letters into meaningless blobs or splits single letters
into multiple "characters" — exactly the fragment-soup seen in Images 4 and 5. On top of that,
our own preprocessing pipeline made things measurably worse for the portrait-oriented book and
newspaper pages: forcing every image into a fixed 512×128 canvas regardless of its original
aspect ratio squashed tall pages so severely that no readable structure survived at all, which is
why Images 1 and 2 returned nothing. Between the script mismatch and the aspect-ratio distortion,
it's clear a dedicated Urdu OCR model — one trained specifically on Nastaliq letterforms, paired
with preprocessing that preserves each image's natural proportions — is genuinely necessary.

## Bonus — Fixing the Aspect-Ratio Bug

The Gap Analysis above correctly diagnosed the biggest problem with this pipeline: `cv2.resize(gray, (512, 128))` **stretches every image to fill the target box**, regardless of its original shape. A tall 510×706 portrait page gets squashed to a fifth of its height before OCR ever sees it — that's why several processed images above look distorted and why Tesseract returned nothing on Images 1 and 2.

The fix is to **preserve the original aspect ratio** and pad the leftover space, instead of stretching. This is called *letterboxing*:
1. Scale the image down (never up) so it fits *within* 512×128, keeping width:height proportions intact.
2. Paste the scaled image onto a blank white 512×128 canvas, centered.
3. Denoise and binarize as before.

This keeps every letterform's true shape, which matters a lot for Nastaliq's diagonal strokes.

In [6]:
def preprocess_image_v2(image_path, save_path, target_size=(512, 128), pad_value=255):
    """Aspect-ratio-preserving version of preprocess_image.
    Scales the image to fit inside target_size, then pads (letterboxes)
    with white space instead of stretching it out of shape.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    target_w, target_h = target_size
    h, w = gray.shape

    # Scale to fit inside the target box, keeping aspect ratio
    scale = min(target_w / w, target_h / h)
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Paste centered onto a blank white canvas (letterbox padding)
    canvas = np.full((target_h, target_w), pad_value, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    denoised = cv2.fastNlMeansDenoising(canvas, h=10)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    cv2.imwrite(save_path, binary)
    return binary


# Reprocess all 153 raw images with the fixed function into a separate folder
# (kept separate from data/processed/ so your original Gap Analysis above still
# matches the images/outputs it describes)
os.makedirs('data/processed_fixed', exist_ok=True)

fixed_count = 0
for img_path in sorted(all_images):
    filename = os.path.basename(img_path)
    save_path = f'data/processed_fixed/{filename}'
    result = preprocess_image_v2(img_path, save_path)
    if result is not None:
        fixed_count += 1

print(f'Done! Re-processed {fixed_count} images with aspect-ratio preserved')
print('Check data/processed_fixed/ folder')

Done! Re-processed 153 images with aspect-ratio preserved
Check data/processed_fixed/ folder
